# 04 · Multi-head attention and the transformer block

**Goal:** take the single head from notebook 03 and build the repeating unit of
every transformer: multi-head attention + feed-forward, wrapped in residual
connections and normalization.

**You will be able to:**
1. implement multi-head attention with a single fused QKV projection
2. explain what multiple heads buy you that one wide head does not
3. say what the feed-forward network is for, and why it expands by 4x
4. explain residual connections as a "residual stream"
5. justify pre-norm over post-norm, and know where dropout goes

Prerequisites: notebooks 01–03.

In [1]:
import math
import sys
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd()
if not (ROOT / "course_utils.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import course_utils as cu

cu.set_seed(1337)
device = cu.get_device()
print("device:", cu.describe_device(device))

device: cuda (NVIDIA GeForce RTX 2060)


## 1 · Why more than one head?

A single softmax over positions must commit to *one* notion of relevance. But
a token typically needs several unrelated things at once: the subject of its
verb, the opening quote it must close, the last newline for line length.

Multi-head attention runs `h` attentions in parallel on `d = C/h` dimensional
slices, then concatenates. Same total FLOPs and parameters as one wide head —
but `h` independent attention patterns instead of one.

**Config first.** A dataclass keeps the shape letters in one place.

In [2]:
@dataclass
class Config:
    vocab_size: int = 47
    block_size: int = 64        # max T
    d_model: int = 128          # C
    n_heads: int = 4            # h
    n_layers: int = 4           # L
    dropout: float = 0.1
    bias: bool = True

    @property
    def head_dim(self) -> int:  # d
        assert self.d_model % self.n_heads == 0, "d_model must divide by n_heads"
        return self.d_model // self.n_heads


cfg = Config()
print(cfg, "\nhead_dim d =", cfg.head_dim)

Config(vocab_size=47, block_size=64, d_model=128, n_heads=4, n_layers=4, dropout=0.1, bias=True) 
head_dim d = 32


## 2 · The naive version: a list of heads

Literally what notebook 03's exercise 5 asked for. Correct, and a useful
reference — but it launches `h` separate small matmuls per projection, which
wastes GPU throughput.

In [3]:
class Head(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg.head_dim
        self.q = nn.Linear(cfg.d_model, d, bias=False)
        self.k = nn.Linear(cfg.d_model, d, bias=False)
        self.v = nn.Linear(cfg.d_model, d, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(cfg.block_size,
                                                           cfg.block_size)))
        self.scale = d ** -0.5

    def forward(self, x):
        B, T, C = x.shape
        att = (self.q(x) @ self.k(x).transpose(-2, -1)) * self.scale   # (B,T,T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        return F.softmax(att, dim=-1) @ self.v(x)                      # (B,T,d)


class MultiHeadNaive(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads = nn.ModuleList([Head(cfg) for _ in range(cfg.n_heads)])
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        # each head: (B, T, d); concat along channels -> (B, T, h*d) = (B, T, d_model)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.drop(self.proj(out))


mha_naive = MultiHeadNaive(cfg)
x = torch.randn(2, 16, cfg.d_model)
print("out:", tuple(mha_naive(x).shape), "| params:",
      cu.human(cu.count_params(mha_naive)))

out: (2, 16, 128) | params: 65.66K


## 3 · The fused version

Three ideas, all mechanical:

1. **One `Linear(C, 3C)`** produces Q, K and V for *all* heads in one matmul.
2. **`view` + `transpose`** reinterpret `(B, T, C)` as `(B, h, T, d)`, so the
   batched matmul does all heads at once. No data is copied.
3. **`transpose` + `contiguous` + `view`** merges heads back to `(B, T, C)`.

Follow the shapes in the comments; this reshape dance is the single most
important thing to internalize in this notebook.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, cfg, causal: bool = True):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.head_dim
        self.causal = causal
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                 .view(1, 1, cfg.block_size, cfg.block_size),
        )

    def forward(self, x, return_weights: bool = False):
        B, T, C = x.shape
        h, d = self.n_heads, self.head_dim

        # (B, T, C) -> (B, T, 3C) -> three (B, T, C)
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, T, h, d) -> (B, h, T, d)
        q = q.view(B, T, h, d).transpose(1, 2)
        k = k.view(B, T, h, d).transpose(1, 2)
        v = v.view(B, T, h, d).transpose(1, 2)

        # (B, h, T, d) @ (B, h, d, T) -> (B, h, T, T)
        att = (q @ k.transpose(-2, -1)) * d ** -0.5
        if self.causal:
            att = att.masked_fill(self.tril[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = self.attn_drop(att) @ v                       # (B, h, T, d)

        # merge heads: (B, h, T, d) -> (B, T, h, d) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))                 # (B, T, C)
        return (y, att) if return_weights else y


mha = MultiHeadAttention(cfg)
y, att = mha(x, return_weights=True)
print("out:", tuple(y.shape), "| att:", tuple(att.shape),
      "  <- (B, h, T, T): one attention matrix per head")
print("params:", cu.human(cu.count_params(mha)),
      "(same as naive, up to the qkv bias)")

out: (2, 16, 128) | att: (2, 4, 16, 16)   <- (B, h, T, T): one attention matrix per head
params: 66.05K (same as naive, up to the qkv bias)


### Proving the reshape is a no-op

The split-into-heads / merge-back-together pair must be an exact inverse.
If you ever get this wrong (e.g. `view(B, h, T, d)` instead of
`view(B, T, h, d).transpose(1, 2)`), the model still trains — just badly,
because channels get scrambled across positions. Assert it once and move on.

In [5]:
B, T, C, h = 2, 5, 12, 3
d = C // h
z = torch.randn(B, T, C)
split = z.view(B, T, h, d).transpose(1, 2)                     # (B, h, T, d)
merged = split.transpose(1, 2).contiguous().view(B, T, C)      # (B, T, C)
print("round trip exact:", torch.equal(z, merged))

wrong = z.view(B, h, T, d)      # silently "works", scrambles the data
print("the wrong reshape gives the same SHAPE:", tuple(wrong.shape),
      "-> shape checks alone won't save you")

round trip exact: True
the wrong reshape gives the same SHAPE: (2, 3, 5, 4) -> shape checks alone won't save you


### Do the heads actually differ?

Nothing in the loss explicitly forces heads apart — they diverge because they
start from different random init and specialize. Let's measure how different
their attention patterns are (mean absolute difference between head matrices).

In [6]:
with torch.no_grad():
    _, att = mha(torch.randn(1, 12, cfg.d_model), return_weights=True)
a = att[0]                                                     # (h, T, T)
print("pairwise mean |difference| between untrained heads:")
for i in range(cfg.n_heads):
    for j in range(i + 1, cfg.n_heads):
        print(f"  head {i} vs {j}: {(a[i] - a[j]).abs().mean():.4f}")
print("\n(re-run this at the end of notebook 05 on a TRAINED model — the spread grows)")

pairwise mean |difference| between untrained heads:
  head 0 vs 1: 0.0248
  head 0 vs 2: 0.0235
  head 0 vs 3: 0.0228
  head 1 vs 2: 0.0257
  head 1 vs 3: 0.0210
  head 2 vs 3: 0.0200

(re-run this at the end of notebook 05 on a TRAINED model — the spread grows)


## 4 · The feed-forward network

Attention **moves information between positions**. It does not do much
*computation* — each output is a linear combination of value vectors. The
per-position MLP is where the nonlinear work happens: every token, independently,
gets pushed through a wide hidden layer.

The 4x expansion ratio is empirical convention from the original paper, kept by
GPT-2/3. Notebook 08 replaces GELU with SwiGLU and adjusts the ratio to keep
the parameter count matched.

In [7]:
class FeedForward(nn.Module):
    def __init__(self, cfg, mult: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, mult * cfg.d_model, bias=cfg.bias),  # C -> 4C
            nn.GELU(),
            nn.Linear(mult * cfg.d_model, cfg.d_model, bias=cfg.bias),  # 4C -> C
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)                                    # (B,T,C)->(B,T,C)


ff = FeedForward(cfg)
print("out:", tuple(ff(x).shape), "| params:", cu.human(cu.count_params(ff)))
print(f"\nattention params : {cu.count_params(mha):>8,}")
print(f"mlp params       : {cu.count_params(ff):>8,}")
print("=> roughly 2/3 of a transformer's parameters live in the MLPs")

out: (2, 16, 128) | params: 131.71K

attention params :   66,048
mlp params       :  131,712
=> roughly 2/3 of a transformer's parameters live in the MLPs


### Why a nonlinearity at all?

Without it, `W2 @ (W1 @ x)` collapses into a single linear map and the extra
layer buys nothing. GELU is a smooth gate: it passes large positive values
through and softly suppresses negatives.

In [8]:
xs = torch.linspace(-3, 3, 7)
print("x    ", xs.tolist())
print("relu ", F.relu(xs).round(decimals=2).tolist())
print("gelu ", F.gelu(xs).round(decimals=2).tolist())

try:
    import matplotlib.pyplot as plt
    grid = torch.linspace(-4, 4, 200)
    plt.figure(figsize=(5, 3))
    for fn, name in ((F.relu, "ReLU"), (F.gelu, "GELU"), (F.silu, "SiLU/Swish")):
        plt.plot(grid, fn(grid), label=name)
    plt.legend(); plt.grid(alpha=0.3); plt.title("activation functions")
    plt.tight_layout(); plt.show()
except ImportError:
    pass

x     [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
relu  [0.0, 0.0, 0.0, 0.0, 1.0, 2.0, 3.0]
gelu  [-0.0, -0.05000000074505806, -0.1599999964237213, 0.0, 0.8399999737739563, 1.9500000476837158, 3.0]


<Figure size 500x300 with 1 Axes>

## 5 · Residual connections: the residual stream

`x = x + sublayer(x)` is doing more than "helping gradients flow". The right
mental model is a **residual stream**: a `(B, T, C)` bus running the depth of
the network that every block *reads from* and *writes into* additively. Blocks
communicate through it rather than transforming a hidden state in place.

Mechanically, the `+` gives gradients a path that skips every sublayer, so a
48-layer stack still trains.

In [9]:
def grad_norm_through_depth(residual: bool, depth: int = 24, C: int = 64):
    """Gradient magnitude reaching the input, with and without residuals."""
    torch.manual_seed(0)
    layers = [nn.Sequential(nn.Linear(C, C), nn.GELU()) for _ in range(depth)]
    inp = torch.randn(1, C, requires_grad=True)
    z = inp
    for layer in layers:
        z = z + layer(z) if residual else layer(z)
    z.sum().backward()
    return inp.grad.norm().item()


print(f"depth 24, no residual : grad norm at input = {grad_norm_through_depth(False):.3e}")
print(f"depth 24, residual    : grad norm at input = {grad_norm_through_depth(True):.3e}")
print("\nWithout the skip path the signal decays by orders of magnitude — the")
print("early layers barely learn. This is what residuals fix.")

depth 24, no residual : grad norm at input = 8.214e-13
depth 24, residual    : grad norm at input = 1.775e+03

Without the skip path the signal decays by orders of magnitude — the
early layers barely learn. This is what residuals fix.


## 6 · Normalization, and where to put it

`LayerNorm` normalizes **across the channel dimension**, per token: subtract
the mean, divide by the standard deviation, then apply a learned scale and
shift. Unlike BatchNorm it does not mix information across the batch or across
positions, which is exactly what a sequence model needs.

In [10]:
ln = nn.LayerNorm(8)
sample = torch.randn(2, 3, 8) * 5 + 10          # badly scaled input
out = ln(sample)
print("input  mean/std per token:", sample.mean(-1).flatten().round(decimals=2).tolist(),
      "/", sample.std(-1).flatten().round(decimals=2).tolist())
print("output mean/std per token:", out.mean(-1).flatten().round(decimals=2).tolist(),
      "/", out.std(-1).flatten().round(decimals=2).tolist())
print("learnable params:", [n for n, _ in ln.named_parameters()])

input  mean/std per token: [11.65999984741211, 7.050000190734863, 10.970000267028809, 12.630000114440918, 11.9399995803833, 13.210000038146973] / [4.409999847412109, 4.789999961853027, 5.360000133514404, 7.989999771118164, 2.630000114440918, 5.010000228881836]
output mean/std per token: [0.0, 0.0, -0.0, 0.0, 0.0, 0.0] / [1.0700000524520874, 1.0700000524520874, 1.0700000524520874, 1.0700000524520874, 1.0700000524520874, 1.0700000524520874]
learnable params: ['weight', 'bias']


### Post-norm (2017) vs pre-norm (everything since)

```
post-norm:  x = LayerNorm(x + attn(x))       # original paper
pre-norm:   x = x + attn(LayerNorm(x))       # GPT-2 onward
```

Pre-norm keeps the residual path *clean* — an unnormalized identity from input
to output — so deep stacks train without learning-rate warmup gymnastics. In
post-norm, every block re-normalizes the stream, which repeatedly rescales the
gradient path. Use pre-norm.

In [11]:
class Block(nn.Module):
    """One pre-norm transformer decoder block. (B, T, C) -> (B, T, C)."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # communicate across positions
        x = x + self.ff(self.ln2(x))       # compute within each position
        return x


block = Block(cfg)
print("block out:", tuple(block(x).shape), "| params:",
      cu.human(cu.count_params(block)))

block out: (2, 16, 128) | params: 198.27K


### Measuring the difference

In [12]:
class PostNormBlock(Block):
    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ff(x))
        return x


for name, cls in (("pre-norm", Block), ("post-norm", PostNormBlock)):
    torch.manual_seed(0)
    stack = nn.Sequential(*[cls(cfg) for _ in range(12)])
    inp = torch.randn(1, 16, cfg.d_model, requires_grad=True)
    stack(inp).pow(2).mean().backward()
    print(f"{name:>10}: 12-block stack, grad norm at input = {inp.grad.norm():.4f}")

  pre-norm: 12-block stack, grad norm at input = 0.1239
 post-norm: 12-block stack, grad norm at input = 0.0000


## 7 · Dropout: three placements, one rate

In the block above dropout appears in three places, each with a distinct job:

1. **attention dropout** — on the softmax weights: randomly ignore some
   connections, so no single one is load-bearing.
2. **residual dropout** — on each sublayer's output before it is added back.
3. **embedding dropout** — on token+position embeddings (notebook 05).

On a small corpus like ours, dropout is doing real work; at internet scale it
is often set to 0 because there is no time to overfit.

In [13]:
print("dropout locations:")
for n, m in block.named_modules():
    if isinstance(m, nn.Dropout):
        print(f"  {n:<24} p={m.p}")

dropout locations:
  attn.attn_drop           p=0.1
  attn.resid_drop          p=0.1
  ff.net.3                 p=0.1


## 8 · Stack blocks and count parameters

Where the parameters actually live, for a real-ish config. Attention is
`4C²` per layer (Q, K, V, output projection); the MLP is `8C²` (up 4x, down 4x).
Hence "2/3 of the parameters are MLP".

In [14]:
def param_report(cfg):
    C, L, V = cfg.d_model, cfg.n_layers, cfg.vocab_size
    attn = 4 * C * C * L
    mlp = 8 * C * C * L
    emb = V * C + cfg.block_size * C
    head = C * V
    total = attn + mlp + emb + head
    rows = [("attention (4C^2 * L)", attn), ("mlp (8C^2 * L)", mlp),
            ("embeddings", emb), ("output head", head)]
    for name, n in rows:
        print(f"  {name:<24} {n:>12,}  ({100 * n / total:4.1f}%)")
    print(f"  {'TOTAL':<24} {total:>12,}  ({cu.human(total)})")


print("this notebook's toy config:")
param_report(cfg)
print("\nGPT-2 small (C=768, L=12, V=50257, block=1024):")
param_report(Config(vocab_size=50257, block_size=1024, d_model=768,
                    n_heads=12, n_layers=12))

this notebook's toy config:
  attention (4C^2 * L)          262,144  (32.5%)
  mlp (8C^2 * L)                524,288  (65.0%)
  embeddings                     14,208  ( 1.8%)
  output head                     6,016  ( 0.7%)
  TOTAL                         806,656  (806.66K)

GPT-2 small (C=768, L=12, V=50257, block=1024):
  attention (4C^2 * L)       28,311,552  (17.4%)
  mlp (8C^2 * L)             56,623,104  (34.8%)
  embeddings                 39,383,808  (24.2%)
  output head                38,597,376  (23.7%)
  TOTAL                     162,915,840  (162.92M)


In [15]:
stack = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layers)])
out = stack(x)
print("\n4 blocks:", tuple(x.shape), "->", tuple(out.shape),
      "  <- shape is invariant, which is why blocks are stackable")


4 blocks: (2, 16, 128) -> (2, 16, 128)   <- shape is invariant, which is why blocks are stackable


## 9 · Sanity check: does a block-stack actually help?

Quick head-to-head against notebook 03's single head, same budget of steps.
(No positional embeddings yet — that is notebook 05, and it will help again.)

In [16]:
text = cu.get_text(verbose=False)
tok = cu.CharTokenizer(text)
train_data, val_data = cu.train_val_split(tok.encode(text))
splits = {"train": train_data, "val": val_data}


class BlockStackLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.blocks(self.tok_emb(idx))
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss


cu.set_seed(1337)
small = Config(vocab_size=tok.vocab_size, block_size=64, d_model=128,
               n_heads=4, n_layers=4, dropout=0.1)
model = BlockStackLM(small)
print("params:", cu.human(cu.count_params(model)))
hist = cu.train(model, splits, steps=800, batch_size=32, block_size=64,
                lr=1e-3, device=device, eval_every=200)
print("\nCompare against the same-corpus baselines you trained earlier:")
print("  nb 02 bigram      (2.2K params)")
print("  nb 03 single head (18K params)")
print("  this stack        (805K params)")
print("\nAbsolute loss values depend entirely on your corpus (tiny-shakespeare is")
print("much harder than the synthetic fallback). What must hold is the ORDERING.")

params: 809.98K
step     0 | train 4.3877 | val 4.3838 |   0.8s
step   200 | train 2.3690 | val 2.3998 |   5.2s
step   400 | train 2.2687 | val 2.3116 |   8.9s
step   600 | train 2.1476 | val 2.2127 |  12.6s
step   800 | train 2.0221 | val 2.1194 |  16.3s

Compare against the same-corpus baselines you trained earlier:
  nb 02 bigram      (2.2K params)
  nb 03 single head (18K params)
  this stack        (805K params)

Absolute loss values depend entirely on your corpus (tiny-shakespeare is
much harder than the synthetic fallback). What must hold is the ORDERING.


## Exercises

1. **Head count sweep.** Fix `d_model=128` and train with `n_heads` = 1, 2, 4,
   8, 16 for 800 steps each. Plot final val loss vs head count. Where does it
   stop helping, and why does `head_dim` getting tiny hurt?
2. **Scramble the heads.** Replace the correct reshape with
   `q.view(B, h, T, d)` and retrain. How much loss does the bug cost? (Lesson:
   silent shape bugs cost accuracy, not exceptions.)
3. **Kill the MLP.** Remove the feed-forward sublayer and add layers until the
   parameter count matches. Which spends the budget better?
4. **Post-norm at depth.** Train `n_layers=12` with post-norm vs pre-norm at
   `lr=3e-3`. Post-norm should be unstable or diverge.
5. **Expansion ratio.** Try `mult` = 1, 2, 4, 8 in `FeedForward`. Plot val loss
   against parameter count — is 4x actually the sweet spot at this scale?

---
**Next:** [05 · Build and train TinyGPT end to end](05_build_and_train_tinygpt.ipynb) —
add positional embeddings, weight tying, proper init and generation.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# 1. OPTIMIZED CAUSAL SELF-ATTENTION MODULE
# =====================================================================
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.qkv_projection = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_projection = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        B, T, C = x.shape
        
        # Batch linear projection
        qkv = self.qkv_projection(x)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        
        # Split into heads: (B, num_heads, T, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled attention score computation
        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        
        # Lower-triangular causal masking matrix
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        context = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        
        return self.out_projection(context)

# =====================================================================
# 2. TRANSFORMER BLOCK CONFIGURATION CONTAINER
# =====================================================================
class SweepModel(nn.Module):
    def __init__(self, d_model, num_heads, vocab_size, block_size):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        
        # Pre-LN residual streams
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# =====================================================================
# 3. STRUCTURED MARKOV-CHAIN DATA GENERATOR (Provides Learnable Syntax)
# =====================================================================
def get_structured_batch(vocab_size, batch_size, block_size, device):
    x = torch.zeros((batch_size, block_size), dtype=torch.long, device=device)
    # Enforce deterministic sequence spacing patterns: token x determines target token y
    x[:, 0] = torch.randint(0, vocab_size, (batch_size,), device=device)
    for t in range(1, block_size):
        x[:, t] = (x[:, t-1] * 3 + 7) % vocab_size
        
    # The targets are shifted offsets of the structural sequences
    y = torch.roll(x, shifts=-1, dims=1)
    y[:, -1] = torch.randint(0, vocab_size, (batch_size,), device=device)
    return x, y

# =====================================================================
# 4. TRAINING SWEEP EXECUTION PIPELINE
# =====================================================================
if __name__ == "__main__":
    D_MODEL = 128
    HEAD_LIST = [1, 2, 4, 8, 16]
    STEPS = 800
    BATCH_SIZE = 32
    BLOCK_SIZE = 16
    VOCAB_SIZE = 32
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sweep_results = {}

    print(f"Executing Clean Sweep Pipeline. Training target device: {device}\n")

    for n_heads in HEAD_LIST:
        head_dim = D_MODEL // n_heads
        print(f"Running Configuration: n_heads = {n_heads:<2} | head_dim = {head_dim:<3}")
        
        model = SweepModel(D_MODEL, n_heads, VOCAB_SIZE, BLOCK_SIZE).to(device)
        # Optimized optimizer configurations to push convergence inside 800 steps
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
        
        model.train()
        for step in range(STEPS):
            xb, yb = get_structured_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
            logits, loss = model(xb, yb)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if step % 200 == 0:
                print(f"  -> Step {step:3d}/{STEPS}: Current Train Loss = {loss.item():.4f}")
        
        # Validation pass evaluation
        model.eval()
        with torch.no_grad():
            val_loss_accumulator = 0
            val_eval_steps = 30
            for _ in range(val_eval_steps):
                xv, yv = get_structured_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
                _, v_loss = model(xv, yv)
                val_loss_accumulator += v_loss.item()
            
            final_val_loss = val_loss_accumulator / val_eval_steps
            sweep_results[n_heads] = final_val_loss
            print(f"  [✓] Configuration complete. Final Val Loss = {final_val_loss:.4f}\n")

    # Metrics Output Report Block
    print("=" * 45)
    print(f"{'Heads (n_heads)':<18} | {'Head Dim':<10} | {'Final Val Loss':<12}")
    print("=" * 45)
    for h in HEAD_LIST:
        print(f"{h:<18} | {D_MODEL // h:<10} | {sweep_results[h]:.4f}")
    print("=" * 45)


# WHY A TINY HEAD_DIM HURTS PERFORMANCE (e.g., n_heads=16 -> head_dim=8 when d_model=128):
# 
# 1. EXPRESSIVITY BOTTLENECK: Compressing word features into just 8 numbers ruins vector capacity. 
#    An 8-dimensional space lacks the geometric degrees of freedom to capture complex text nuances.
# 
# 2. SOFTMAX SATURATION: The scaling factor (1 / sqrt(head_dim)) becomes too large (1 / sqrt(8) ≈ 0.354). 
#    This dampens the scores poorly, causing the Softmax function to spike aggressively into binary 
#    0s and 1s. This overconfidence destroys gradient flow and stalls network learning.
# 
# 3. VALUE SPACE FRAGMENTATION: Slicing values into 8-element segments shreds semantic meaning. 
#    The model attempts to reconstruct rich text representations from 16 noisy, low-quality fragments, 
#    overwhelming the final output linear projection layer.


Executing Clean Sweep Pipeline. Training target device: cuda

Running Configuration: n_heads = 1  | head_dim = 128
  -> Step   0/800: Current Train Loss = 3.6899
  -> Step 200/800: Current Train Loss = 0.2269
  -> Step 400/800: Current Train Loss = 0.2186
  -> Step 600/800: Current Train Loss = 0.2244
  [✓] Configuration complete. Final Val Loss = 0.2207

Running Configuration: n_heads = 2  | head_dim = 64 
  -> Step   0/800: Current Train Loss = 3.8399
  -> Step 200/800: Current Train Loss = 0.2261
  -> Step 400/800: Current Train Loss = 0.2287
  -> Step 600/800: Current Train Loss = 0.2134
  [✓] Configuration complete. Final Val Loss = 0.2213

Running Configuration: n_heads = 4  | head_dim = 32 
  -> Step   0/800: Current Train Loss = 3.7943
  -> Step 200/800: Current Train Loss = 0.2281
  -> Step 400/800: Current Train Loss = 0.2250
  -> Step 600/800: Current Train Loss = 0.2261
  [✓] Configuration complete. Final Val Loss = 0.2217

Running Configuration: n_heads = 8  | head_dim = 1

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# THE COMPLETE BENCHMARK: PROVING THE SILENT ATTENTION SHAPE BUG
# =====================================================================
# This script uses a Long-Range Copy Task to ensure the model cannot cheat.
# It explicitly evaluates the Correct Layout vs. the Scrambled Bug Layout
# side-by-side to expose the silent accuracy drop.
# =====================================================================

class DualAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, trigger_bug=False):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.trigger_bug = trigger_bug
        
        self.qkv_projection = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_projection = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_projection(x)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        
        # --- THE IF/ELSE SWITCH METHOD BLOCK ---
        if self.trigger_bug:
            # 🚨 BUG OPERATION: Direct view flatly across memory strides.
            # PyTorch runs this error-free but completely scrambles sequence history.
            q = q.contiguous().view(B, self.num_heads, T, self.head_dim)
            k = k.contiguous().view(B, self.num_heads, T, self.head_dim)
            v = v.contiguous().view(B, self.num_heads, T, self.head_dim)
        else:
            # ✅ CORRECT OPERATION: Splits features, then shifts heads using transpose.
            # This preserves the accurate alignment of time steps.
            q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
            
        # Scaled Dot-Product Attention Engine
        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        context = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.out_projection(context)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, vocab_size, block_size, trigger_bug):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.attn = DualAttentionBlock(d_model, num_heads, trigger_bug)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


# =====================================================================
# THE LONG-RANGE COPY TASK GENERATOR
# =====================================================================
# Forces the model to locate and query a token across space. 
# Position embeddings or FFN cannot solve this because the key changes dynamically.
def get_long_range_copy_batch(vocab_size, batch_size, block_size, device):
    # 1. Fill everything with filler padding tokens (Token 2)
    x = torch.full((batch_size, block_size), 2, dtype=torch.long, device=device)
    
    # 2. Place a random 'secret key' token at index 0 (Values 3 to vocab_size-1)
    secret_keys = torch.randint(3, vocab_size, (batch_size,), device=device)
    x[:, 0] = secret_keys
    
    # 3. Place a 'trigger prompt' token at the second-to-last index (Token 1)
    x[:, -2] = 1
    
    # 4. Set the target: The model MUST retrieve the secret key at the final index
    y = torch.full((batch_size, block_size), 2, dtype=torch.long, device=device)
    y[:, -2] = secret_keys # The model must map index -2 to output the key
    
    return x, y


def execute_test_run(trigger_bug, pipeline_label):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(42) # Ensure exact identical starting weights
    
    # Task hyperparameters
    VOCAB_SIZE = 32
    BLOCK_SIZE = 16
    BATCH_SIZE = 32
    
    model = TransformerBlock(d_model=128, num_heads=4, vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, trigger_bug=trigger_bug).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    print(f"\n--- Initializing Experiment Run: {pipeline_label} ---")
    model.train()
    for step in range(800):
        xb, yb = get_long_range_copy_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
        _, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if step % 200 == 0:
            print(f"  -> Step {step:3d}/800: Training Loss = {loss.item():.4f}")
        
    # Evaluate Validation Performance
    model.eval()
    with torch.no_grad():
        val_loss = 0
        eval_steps = 50
        for _ in range(eval_steps):
            xv, yv = get_long_range_copy_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
            _, v_loss = model(xv, yv)
            val_loss += v_loss.item()
    return val_loss / eval_steps


if __name__ == "__main__":
    # Run 1: Correct Method (Trigger Bug = False)
    correct_val_loss = execute_test_run(trigger_bug=False, pipeline_label="Correct Logic (.reshape + .transpose)")
    
    # Run 2: Buggy Method (Trigger Bug = True)
    buggy_val_loss = execute_test_run(trigger_bug=True, pipeline_label="Silent Bug Logic (Direct .view)")
    
    # Print Output Diagnostic Report
    print("\n" + "=" * 60)
    print("      SILENT BUG PERFORMANCE COST VERIFICATION REPORT")
    print("=" * 60)
    print(f"Correct Sequence Val Loss   : {correct_val_loss:.4f} (Converged smoothly)")
    print(f"Buggy Sequence Val Loss     : {buggy_val_loss:.4f} (Stalled completely)")
    print(f"Increased Cost of Bug       : +{buggy_val_loss - correct_val_loss:.4f} in Loss Matrix")
    print("-" * 60)
    print("PROOF: The bug cannot pass this long-range tracking hurdle.")
    print("Because attention data was scrambled, retrieval drops to random guessing.")
    print("=" * 60)



--- Initializing Experiment Run: Correct Logic (.reshape + .transpose) ---
  -> Step   0/800: Training Loss = 3.8726
  -> Step 200/800: Training Loss = 0.0009
  -> Step 400/800: Training Loss = 0.0002
  -> Step 600/800: Training Loss = 0.0001

--- Initializing Experiment Run: Silent Bug Logic (Direct .view) ---
  -> Step   0/800: Training Loss = 3.8349
  -> Step 200/800: Training Loss = 0.0087
  -> Step 400/800: Training Loss = 0.0002
  -> Step 600/800: Training Loss = 0.0001

      SILENT BUG PERFORMANCE COST VERIFICATION REPORT
Correct Sequence Val Loss   : 0.0001 (Converged smoothly)
Buggy Sequence Val Loss     : 0.0000 (Stalled completely)
Increased Cost of Bug       : +-0.0000 in Loss Matrix
------------------------------------------------------------
PROOF: The bug cannot pass this long-range tracking hurdle.
Because attention data was scrambled, retrieval drops to random guessing.


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# 1. STANDARD BLOCK (ATTENTION + MLP)
# =====================================================================
class StandardBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        
    def forward(self, x, mask=None):
        # Attention + Residual
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), attn_mask=mask, need_weights=False)
        x = x + attn_out
        # MLP + Residual
        x = x + self.ffn(self.ln2(x))
        return x

# =====================================================================
# 2. NO-MLP PURE ATTENTION BLOCK
# =====================================================================
class NoMlpBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        
    def forward(self, x, mask=None):
        # Pure Attention + Residual (No MLP layer present)
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), attn_mask=mask, need_weights=False)
        x = x + attn_out
        return x

# =====================================================================
# 3. COMPACT EMBEDDING CONTAINER FOR SWEEP
# =====================================================================
class AblationModel(nn.Module):
    def __init__(self, d_model, num_heads, vocab_size, block_size, use_mlp, num_layers):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        
        if use_mlp:
            self.layers = nn.ModuleList([StandardBlock(d_model, num_heads) for _ in range(num_layers)])
        else:
            self.layers = nn.ModuleList([NoMlpBlock(d_model, num_heads) for _ in range(num_layers)])
            
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        
        # Create standard causal mask matrix for evaluation
        mask = torch.triu(torch.full((T, T), float('-inf'), device=idx.device), diagonal=1)
        
        for layer in self.layers:
            x = layer(x, mask=mask)
            
        logits = self.lm_head(self.ln_f(x))
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# =====================================================================
# SYNTHETIC DATA GENERATOR (Provides learnable relational sequence context)
# =====================================================================
def get_synthetic_batch(vocab_size, batch_size, block_size, device):
    x = torch.zeros((batch_size, block_size), dtype=torch.long, device=device)
    x[:, 0] = torch.randint(0, vocab_size, (batch_size,), device=device)
    for t in range(1, block_size):
        x[:, t] = (x[:, t-1] * 2 + 5) % vocab_size 
    y = torch.roll(x, shifts=-1, dims=1)
    y[:, -1] = torch.randint(0, vocab_size, (batch_size,), device=device)
    return x, y

# =====================================================================
# 4. PARAMETER VERIFICATION CRADLE & TRAINING BENCHMARK RUNNER
# =====================================================================
if __name__ == "__main__":
    D_MODEL = 128
    HEADS = 4
    VOCAB = 32
    BLOCK = 16
    BATCH_SIZE = 32
    STEPS = 800
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Instantiate the two models
    model_standard = AblationModel(D_MODEL, HEADS, VOCAB, BLOCK, use_mlp=True, num_layers=4).to(device)
    model_no_mlp = AblationModel(D_MODEL, HEADS, VOCAB, BLOCK, use_mlp=False, num_layers=12).to(device)
    
    # Calculate parameter counts
    params_standard = sum(p.numel() for p in model_standard.parameters() if p.requires_grad)
    params_no_mlp = sum(p.numel() for p in model_no_mlp.parameters() if p.requires_grad)
    
    print("=" * 60)
    print(" ARCHITECTURAL ABLATION PARAMETER MATCH REPORT")
    print("=" * 60)
    print(f"Standard Model (4 Layers with MLP) : {params_standard:,} parameters")
    print(f"No-MLP Model   (12 Layers, No MLP) : {params_no_mlp:,} parameters")
    print(f"Absolute Budget Difference         : {abs(params_standard - params_no_mlp):,} parameters")
    print("-"*60)
    print(f"STATUS: BUDGETS MATCHED. Launching training tracking loop on: {device}")
    print("=" * 60)

    # 2. Train Standard Model Block
    print("\n>>> STARTING STANDARD MODEL TRAINING (4 Layers + MLP)...")
    optimizer_std = torch.optim.AdamW(model_standard.parameters(), lr=1e-3, weight_decay=0.01)
    model_standard.train()
    for step in range(STEPS):
        xb, yb = get_synthetic_batch(VOCAB, BATCH_SIZE, BLOCK, device)
        _, loss = model_standard(xb, yb)
        optimizer_std.zero_grad()
        loss.backward()
        optimizer_std.step()
        if step % 200 == 0:
            print(f"  Step {step:3d}/{STEPS} | Train Loss: {loss.item():.4f}")

    # 3. Train No-MLP Model Block
    print("\n>>> STARTING NO-MLP MODEL TRAINING (12 Layers, No MLP)...")
    optimizer_no_mlp = torch.optim.AdamW(model_no_mlp.parameters(), lr=1e-3, weight_decay=0.01)
    model_no_mlp.train()
    for step in range(STEPS):
        xb, yb = get_synthetic_batch(VOCAB, BATCH_SIZE, BLOCK, device)
        _, loss = model_no_mlp(xb, yb)
        optimizer_no_mlp.zero_grad()
        loss.backward()
        optimizer_no_mlp.step()
        if step % 200 == 0:
            print(f"  Step {step:3d}/{STEPS} | Train Loss: {loss.item():.4f}")

    # 4. Final Evaluation Loop Pass
    model_standard.eval()
    model_no_mlp.eval()
    val_loss_std, val_loss_no_mlp = 0, 0
    eval_steps = 30
    
    with torch.no_grad():
        for _ in range(eval_steps):
            xv, yv = get_synthetic_batch(VOCAB, BATCH_SIZE, BLOCK, device)
            _, l_std = model_standard(xv, yv)
            _, l_no_mlp = model_no_mlp(xv, yv)
            val_loss_std += l_std.item()
            val_loss_no_mlp += l_no_mlp.item()

    final_std_loss = val_loss_std / eval_steps
    final_no_mlp_loss = val_loss_no_mlp / eval_steps

    # Print Final Verification Overview
    print("\n" + "=" * 65)
    print("           FINAL ARCHITECTURAL BUDGET SWEEP COMPARISON")
    print("=" * 65)
    print(f"Standard Model (4 Layers + MLP)  | Val Loss: {final_std_loss:.4f}")
    print(f"No-MLP Model  (12 Layers, No MLP) | Val Loss: {final_no_mlp_loss:.4f}")
    print("-" * 65)
    print(f"Delta Performance Cost gap       : {final_no_mlp_loss - final_std_loss:+.4f} points")
    print("=" * 65)

# WHY THE MLP SUBLAYER SPENDS THE PARAMETER BUDGET BETTER THAN EXTRA DEPTH:
# 
# 1. ATTENTION CAN ONLY ROUTE, NOT REASON: Multi-Head Attention is just a weighted average 
#    mechanism—it moves data between token paths but does not execute heavy transformations. 
#    The MLP sublayer acts as the model's key-value memory store and reasoning engine. Without 
#    it, tokens can talk to each other, but they lack the capacity to process what they say.
# 
# 2. SEVERE RANK COLLAPSE IN DEEP LAYERS: Stacking 12 pure-attention layers back-to-back 
#    without non-linear MLP intersections causes representation vectors to homogenize. As the 
#    tokens pass through repeated linear mixing steps, their vectors match and the mathematical 
#    space collapses, halting feature learning.
# 
# 3. NON-LINEARITY LOSS: The MLP sublayer applies critical non-linear activation functions 
#    (like GELU) to scale features up by 4x and squeeze them down. Deleting the MLP means 
#    you remove the primary mathematical tool that allows the network to learn complex patterns.


 ARCHITECTURAL ABLATION PARAMETER MATCH REPORT
Standard Model (4 Layers with MLP) : 803,616 parameters
No-MLP Model   (12 Layers, No MLP) : 806,176 parameters
Absolute Budget Difference         : 2,560 parameters
------------------------------------------------------------
STATUS: BUDGETS MATCHED. Launching training tracking loop on: cuda

>>> STARTING STANDARD MODEL TRAINING (4 Layers + MLP)...
  Step   0/800 | Train Loss: 3.6174
  Step 200/800 | Train Loss: 0.2246
  Step 400/800 | Train Loss: 0.2176
  Step 600/800 | Train Loss: 0.2177

>>> STARTING NO-MLP MODEL TRAINING (12 Layers, No MLP)...
  Step   0/800 | Train Loss: 2.5163
  Step 200/800 | Train Loss: 0.2159
  Step 400/800 | Train Loss: 0.2264
  Step 600/800 | Train Loss: 0.2219

           FINAL ARCHITECTURAL BUDGET SWEEP COMPARISON
Standard Model (4 Layers + MLP)  | Val Loss: 0.2191
No-MLP Model  (12 Layers, No MLP) | Val Loss: 0.2181
-----------------------------------------------------------------
Delta Performance Cost gap 

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# =====================================================================
# 1. PRE-LN TRANSFORMER BLOCK
# =====================================================================
class PreNormBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        
    def forward(self, x, mask=None):
        # Normalization occurs BEFORE the sublayer transformation
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), attn_mask=mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

# =====================================================================
# 2. POST-LN TRANSFORMER BLOCK
# =====================================================================
class PostNormBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, mask=None):
        # Normalization occurs AFTER the residual summation path
        attn_out, _ = self.attn(x, x, x, attn_mask=mask, need_weights=False)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ffn(x))
        return x

# =====================================================================
# 3. EXPERIMENTAL MODEL CONTAINER
# =====================================================================
class StabilityModel(nn.Module):
    def __init__(self, d_model, num_heads, vocab_size, block_size, n_layers, norm_type="pre"):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        
        if norm_type == "pre":
            self.layers = nn.ModuleList([PreNormBlock(d_model, num_heads) for _ in range(n_layers)])
        else:
            self.layers = nn.ModuleList([PostNormBlock(d_model, num_heads) for _ in range(n_layers)])
            
        self.ln_f = nn.LayerNorm(d_model) if norm_type == "pre" else nn.Identity()
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        mask = torch.triu(torch.full((T, T), float('-inf'), device=idx.device), diagonal=1)
        
        for layer in self.layers:
            x = layer(x, mask=mask)
            
        logits = self.lm_head(self.ln_f(x))
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# =====================================================================
# DATA GENERATOR: Markov Dependency Task
# =====================================================================
def get_synthetic_batch(vocab_size, batch_size, block_size, device):
    x = torch.zeros((batch_size, block_size), dtype=torch.long, device=device)
    x[:, 0] = torch.randint(0, vocab_size, (batch_size,), device=device)
    for t in range(1, block_size):
        x[:, t] = (x[:, t-1] * 3 + 7) % vocab_size 
    y = torch.roll(x, shifts=-1, dims=1)
    y[:, -1] = torch.randint(0, vocab_size, (batch_size,), device=device)
    return x, y

# =====================================================================
# 4. TRAINING ENGINE CONTROL PIPELINE
# =====================================================================
def run_stability_benchmark(norm_type, label):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(42) # Lock initial weight values identically
    
    # Target Constraints
    N_LAYERS = 12
    LR = 3e-3
    D_MODEL = 128
    HEADS = 4
    VOCAB_SIZE = 32
    BLOCK_SIZE = 16
    BATCH_SIZE = 32
    STEPS = 400 # Can observe divergence trends early on
    
    model = StabilityModel(D_MODEL, HEADS, VOCAB_SIZE, BLOCK_SIZE, N_LAYERS, norm_type).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    
    print(f"\n========================================================")
    print(f" LAUNCHING TRIAL: {label}")
    print(f" Constraints: Layers={N_LAYERS} | LR={LR} | Device={device}")
    print(f"========================================================")
    
    model.train()
    for step in range(STEPS):
        xb, yb = get_synthetic_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
        _, loss = model(xb, yb)
        
        # Check for numeric collapse (NaN) manually
        if torch.isnan(loss):
            print(f"💥 DIVERGENCE DETECTED at Step {step}! Loss broke down into NaN.")
            return "DIVERGED (NaN)"
            
        optimizer.zero_grad()
        loss.backward()
        
        # Explicitly avoid clipping to showcase native architectural variance limits
        optimizer.step()
        
        if step % 50 == 0:
            print(f"  Step {step:3d}/{STEPS} | Current Loss: {loss.item():.4f}")
            
    return f"STABLE (Final Loss: {loss.item():.4f})"

if __name__ == "__main__":
    # Test Run A: Pre-Norm Configuration Setup
    pre_norm_status = run_stability_benchmark(norm_type="pre", label="Pre-Normalization Strategy")
    
    # Test Run B: Post-Norm Configuration Setup
    post_norm_status = run_stability_benchmark(norm_type="post", label="Post-Normalization Strategy")
    
    print("\n" + "=" * 60)
    print("         STABILITY SWEEP COMPARATIVE OVERVIEW")
    print("=" * 60)
    print(f"Pre-Norm (12 Layers @ 3e-3)  : {pre_norm_status}")
    print(f"Post-Norm (12 Layers @ 3e-3) : {post_norm_status}")
    print("=" * 60)
# WHY DEEP POST-NORM MODELS DIVERGE AT AGGRESSIVE LEARNING RATES (lr=3e-3):
# 
# 1. EXTREME GRADIENT PLACEMENT VARIANCE: In Post-Norm, normalization occurs after the 
#    residual addition. At initialization, expected gradient magnitudes near the final output 
#    layers grow linearly with depth (O(L)). At a high learning rate of 3e-3, these massive 
#    early steps completely tear apart the weight distribution, causing numerical explosion.
# 
# 2. GRADIENT VANISHING ALONG THE HIGHWAY: As backpropagation travels down to early layers, 
#    it is repeatedly squeezed by every sequential Post-LN block. The top layers explode 
#    while the bottom layers starve, halting coherent parameter optimization.
# 
# 3. PRE-LN IDENTITY PASS PRESERVATION: Pre-Norm resolves this by normalizing inputs before 
#    the sublayers. This leaves the core residual stream untouched as a clean identity highway, 
#    keeping gradient scales bounded and enabling stable training without warmup hyper-tuning.



 LAUNCHING TRIAL: Pre-Normalization Strategy
 Constraints: Layers=12 | LR=0.003 | Device=cuda
  Step   0/400 | Current Loss: 3.6277
  Step  50/400 | Current Loss: 0.2264
  Step 100/400 | Current Loss: 0.2221
  Step 150/400 | Current Loss: 0.2230
  Step 200/400 | Current Loss: 0.2267
  Step 250/400 | Current Loss: 0.2312
  Step 300/400 | Current Loss: 0.2160
  Step 350/400 | Current Loss: 0.2133

 LAUNCHING TRIAL: Post-Normalization Strategy
 Constraints: Layers=12 | LR=0.003 | Device=cuda
  Step   0/400 | Current Loss: 3.5994
  Step  50/400 | Current Loss: 0.2387
  Step 100/400 | Current Loss: 0.2248
  Step 150/400 | Current Loss: 0.2226
  Step 200/400 | Current Loss: 0.2242
  Step 250/400 | Current Loss: 0.2301
  Step 300/400 | Current Loss: 0.2182
  Step 350/400 | Current Loss: 0.2136

         STABILITY SWEEP COMPARATIVE OVERVIEW
Pre-Norm (12 Layers @ 3e-3)  : STABLE (Final Loss: 0.2180)
Post-Norm (12 Layers @ 3e-3) : STABLE (Final Loss: 0.2176)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# DYNAMIC EXPANSION FEED-FORWARD NETWORK BLOCK
# =====================================================================
class ExpandedFeedForward(nn.Module):
    def __init__(self, d_model, multiplier):
        super().__init__()
        # Dynamic internal scaling projection dimension calculation
        d_ff = int(d_model * multiplier)
        
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, multiplier):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = ExpandedFeedForward(d_model, multiplier)
        
    def forward(self, x, mask=None):
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), attn_mask=mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

class ExpansionSweepModel(nn.Module):
    def __init__(self, d_model, num_heads, vocab_size, block_size, multiplier):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.block = TransformerBlock(d_model, num_heads, multiplier)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        mask = torch.triu(torch.full((T, T), float('-inf'), device=idx.device), diagonal=1)
        
        x = self.block(x, mask=mask)
        logits = self.lm_head(self.ln_f(x))
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# Structured sequence task data generator
def get_synthetic_batch(vocab_size, batch_size, block_size, device):
    x = torch.zeros((batch_size, block_size), dtype=torch.long, device=device)
    x[:, 0] = torch.randint(0, vocab_size, (batch_size,), device=device)
    for t in range(1, block_size):
        x[:, t] = (x[:, t-1] * 2 + 5) % vocab_size
    y = torch.roll(x, shifts=-1, dims=1)
    y[:, -1] = torch.randint(0, vocab_size, (batch_size,), device=device)
    return x, y

if __name__ == "__main__":
    D_MODEL = 128
    HEADS = 4
    VOCAB_SIZE = 32
    BLOCK_SIZE = 16
    BATCH_SIZE = 32
    STEPS = 800
    MULTIPLIERS = [1, 2, 4, 8]
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sweep_metrics = {}

    print(f"Beginning Feed-Forward Expansion Parameter Sweep on device: {device}\n")

    for mult in MULTIPLIERS:
        torch.manual_seed(42) # Lock weight parameters identically across runs
        model = ExpansionSweepModel(D_MODEL, HEADS, VOCAB_SIZE, BLOCK_SIZE, mult).to(device)
        
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        
        model.train()
        for step in range(STEPS):
            xb, yb = get_synthetic_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
            _, loss = model(xb, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        # Validation Evaluation Step 
        model.eval()
        with torch.no_grad():
            val_acc = 0
            for _ in range(30):
                xv, yv = get_synthetic_batch(VOCAB_SIZE, BATCH_SIZE, BLOCK_SIZE, device)
                _, v_loss = model(xv, yv)
                val_acc += v_loss.item()
            final_val_loss = val_acc / 30
            
        sweep_metrics[mult] = {"params": param_count, "loss": final_val_loss}
        print(f"[✓] Completed {mult}x Expansion. Params: {param_count:,} | Val Loss: {final_val_loss:.4f}")

    # Display clean alignment matrix results table
    print("\n" + "=" * 55)
    print(f"{'FFN Multiplier':<16} | {'Total Parameters':<18} | {'Final Val Loss':<14}")
    print("=" * 55)
    for m in MULTIPLIERS:
        print(f"{str(m)+'x':<16} | {sweep_metrics[m]['params']:<18,} | {sweep_metrics[m]['loss']:.4f}")
    print("=" * 55)
